In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import joblib

In [43]:
# 1. Load data
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [44]:
df.sample(5)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
5990,1472-TNCWL,Male,0,No,Yes,36,Yes,No,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,94.70,3512.5,No
3520,6036-TTFYU,Female,0,Yes,No,16,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Mailed check,19.60,314.45,No
1649,2851-MMUTZ,Female,0,No,No,27,Yes,No,DSL,No,...,No,Yes,No,No,Month-to-month,Yes,Mailed check,56.15,1439.35,No
2096,0761-AETCS,Female,0,No,No,1,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Electronic check,19.30,19.3,Yes
6595,5702-KVQRD,Male,0,Yes,No,71,Yes,No,DSL,Yes,...,Yes,Yes,Yes,Yes,Two year,Yes,Electronic check,82.55,5832.65,No


In [45]:
# 2. Basic cleaning
df = df.dropna()
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")  # common gotcha in this dataset
df = df.dropna()

In [46]:
# 3. Encode categorical columns
df = df.drop("customerID", axis=1) 
df = pd.get_dummies(df, drop_first=True)

In [47]:
# 4. Split features/target
X = df.drop("Churn_Yes", axis=1)   # after get_dummies, target becomes Churn_Yes
joblib.dump(X.columns.tolist(), "columns.pkl")
y = df["Churn_Yes"]

In [48]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [49]:
# 5. Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [50]:
# 6. Train (Logistic Regression as baseline)
# model = LogisticRegression(max_iter=1000)
model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(X_train_scaled, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

In [51]:
# 7. Evaluate
preds = model.predict(X_test_scaled)
print("Accuracy:", accuracy_score(y_test, preds))
print("Precision:", precision_score(y_test, preds))
print("Recall:", recall_score(y_test, preds))
print("F1:", f1_score(y_test, preds))
print("Confusion Matrix:\n", confusion_matrix(y_test, preds))

Accuracy: 0.7263681592039801
Precision: 0.49093904448105435
Recall: 0.7967914438502673
F1: 0.6075433231396534
Confusion Matrix:
 [[724 309]
 [ 76 298]]


In [52]:
# 8. Save
joblib.dump(model, "model.pkl")
joblib.dump(scaler, "scaler.pkl")

['scaler.pkl']

In [53]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
knn_preds = knn.predict(X_test_scaled)
print("KNN F1:", f1_score(y_test, knn_preds))

KNN F1: 0.535475234270415


In [54]:
import numpy as np
coefs = pd.Series(model.coef_[0], index=X.columns).sort_values(key=abs, ascending=False)
print(coefs.head(10))

tenure                           -1.263854
MonthlyCharges                   -0.928682
InternetService_Fiber optic       0.746144
Contract_Two year                -0.638598
TotalCharges                      0.623279
Contract_One year                -0.326693
StreamingTV_Yes                   0.259691
StreamingMovies_Yes               0.252870
MultipleLines_Yes                 0.202763
PaymentMethod_Electronic check    0.193697
dtype: float64


In [55]:
df.shape

(7032, 31)